# TFM BraTS-GLI - Baseline fuerte nnU-Net v2

Entrena nnU-Net v2 como baseline externo (pilar 1), sobre el MISMO split del TFM, y evalua con el `evaluate` del TFM para metricas comparables (Dice/HD95 por ET/TC/WT). Detalle y decisiones: `docs/nnunet-baseline.md`.

**GPU recomendada: A100** (nnU-Net es pesado). Comandos en una sola linea (Colab no une `!cmd \`).

**Aviso de disco:** el gran consumidor es `nnUNet_preprocessed` (~130 MB/caso x 1135 ~= 150 GB). Por eso NO se copia el dataset a `/content` (se lee desde Drive por symlink, una sola vez en el preprocesado) y hay celdas `df -h` para vigilar. Si el disco se llena, usa un runtime con mas disco.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar el repo (privado -> token)
Requiere el secret `GITHUB_TOKEN` en Colab (Secrets, icono de la llave). Ver seccion 3 del notebook de Swin.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
Path("/content/git_askpass.py").write_text(
    "#!/usr/bin/env python3\n"
    "import os, sys\n"
    "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['GITHUB_TOKEN'])\n",
    encoding="utf-8",
)
os.chmod("/content/git_askpass.py", 0o700)
os.environ["GIT_ASKPASS"] = "/content/git_askpass.py"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
print("credenciales listas")

In [ ]:
%cd /content
!git clone https://github.com/jesusferron/tfm-brain-tumor-segmentation.git || (cd tfm-brain-tumor-segmentation && git pull origin main)
%cd /content/tfm-brain-tumor-segmentation
!git log --oneline -3

## 4. Instalar dependencias (protocolo + nnU-Net v2)

In [ ]:
%cd /content/tfm-brain-tumor-segmentation
!pip install -r requirements/protocol.txt
!pip install -r requirements/nnunet.txt
!python -c "import torch, nnunetv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('nnunetv2', nnunetv2.__version__)"

## 5. Apuntar `dataset_root` a Drive
No se copia el dataset a `/content` para conservar disco (el preprocesado de nnU-Net ya es grande). Ajusta la ruta si tu carpeta en Drive se llama distinto.

In [ ]:
from pathlib import Path
import yaml

dataset_root = "/content/drive/MyDrive/TFM-datasets"
config_path = Path("configs/dataset/brats_gli_2024.yaml")
config = yaml.safe_load(config_path.read_text())
config["dataset_root"] = dataset_root
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
for name in config["training_roots"]:
    p = Path(dataset_root) / name
    print(p, "OK" if p.is_dir() else "MISSING")

## 6. Convertir los splits del TFM al formato nnU-Net
Symlinks con naming de canal de nnU-Net; preserva etiquetas 0-4; `imagesTs` = split `val` (held out).

In [ ]:
!python scripts/nnunet/prepare_brats_gli_nnunet_full.py --dataset-config configs/dataset/brats_gli_2024.yaml --split-dir outputs/splits/brats_gli_2024_seed20260526 --train-split train.csv --test-split val.csv --nnunet-raw /content/nnUNet_raw --dataset-id 725 --dataset-name BraTSGLI2024 --link-mode symlink

## 7. Variables de entorno de nnU-Net (disco local del runtime)

In [ ]:
import os
os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"
print("nnU-Net env listo")

## 8. Planificar y preprocesar (con verificacion de integridad)
Lee las imagenes una vez (desde Drive por symlink) y escribe el preprocesado en `/content/nnUNet_preprocessed`. Es la fase que mas disco consume.

In [ ]:
!df -h /content
!nnUNetv2_plan_and_preprocess -d 725 --verify_dataset_integrity
!df -h /content

## 9. Entrenar 3d_fullres, fold 0 (250 epocas)
Fase larga (varias horas en A100). nnU-Net valida internamente sobre el 20% del `train` y guarda `checkpoint_best.pth`. Para el nnU-Net canonico (1000 epocas) omite `-tr`.

In [ ]:
!nnUNetv2_train 725 3d_fullres 0 -tr nnUNetTrainer_250epochs

## 10. Predecir sobre `val` (held out)

In [ ]:
!nnUNetv2_predict -i /content/nnUNet_raw/Dataset725_BraTSGLI2024/imagesTs -o /content/nnunet_pred_val -d 725 -c 3d_fullres -f 0 -tr nnUNetTrainer_250epochs -chk checkpoint_best.pth

## 11. Evaluar con el `evaluate` del TFM (metricas comparables)

In [ ]:
!python -m tfm_brats.cli evaluate --dataset-config configs/dataset/brats_gli_2024.yaml --split-csv outputs/splits/brats_gli_2024_seed20260526/val.csv --predictions-dir /content/nnunet_pred_val --output-csv outputs/evaluation/nnunet_3dfullres_val_metrics.csv --output-json outputs/evaluation/nnunet_3dfullres_val_metrics_summary.json

In [ ]:
!cat outputs/evaluation/nnunet_3dfullres_val_metrics_summary.json

## 12. Guardar resultados en Google Drive
Persisten aunque el runtime se desconecte. Incluye el modelo entrenado de nnU-Net.

In [ ]:
import shutil, os
dst = "/content/drive/MyDrive/TFM-resultados/nnunet_3dfullres"
os.makedirs(dst, exist_ok=True)
for f in [
    "outputs/evaluation/nnunet_3dfullres_val_metrics.csv",
    "outputs/evaluation/nnunet_3dfullres_val_metrics_summary.json",
]:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(dst, os.path.basename(f))); print("copiado ->", os.path.basename(f))
    else:
        print("(aun no existe)", f)
shutil.copytree("/content/nnUNet_results/Dataset725_BraTSGLI2024", dst+"/nnUNet_results", dirs_exist_ok=True)
print("\nGuardado en Drive:", dst)

## Que mandarme al terminar
```
GPU (nvidia-smi):
plan_and_preprocess sin error: si/no
Entrenamiento: epocas completadas / se corto:
df -h /content tras preprocesar y tras entrenar:
Contenido de nnunet_3dfullres_val_metrics_summary.json:
```

Evaluacion sobre `test`: solo con la config congelada, re-ejecutando el paso 6 con `--test-split test.csv`.